# Unified Dynamic PBR Reactor Model

**Double BSc BEP (Applied Mathematics & Applied Physics)**  
**Focus:** Catalytic Semi-Hydrogenation of Acetylene ($\text{C}_2\text{H}_2 \to \text{C}_2\text{H}_4 \to \text{C}_2\text{H}_6$) in a Packed Bed Reactor under Dynamic Forcing.

---

## Architecture & Scope

This notebook is the progressive, unified implementation of the packed-bed reactor model. It bridges the literature milestones into a single configurable simulation engine:
1. **Berger (2008)**: Pseudo-homogeneous baseline, instant intraparticle diffusion, dynamic surface coverages $\theta(t)$.
2. **Urmès (2019)**: Heterogeneous PBR, conservative intraparticle pore diffusion, PSSA algebraic surface closure $\theta^*(c_p)$.
3. **Urmès (2022)**: Full heterogeneous PBR, finite pore diffusion, dynamic surface site balances $\frac{\partial \theta}{\partial t} = R_{surf}$.

### Physical Inventory & Map Crosswalk

```text
Inlet Waveform [sc_manipulated_input]
       │  C_i,in(t)
       ▼
Bulk Void-Gas Advection [ma_gas_pde]
       │  Film flux j_i = k_g a (C_b - c_p,shell) [external_film_transfer]
       ▼
Pore Diffusion in Catalyst Pellet [ph_pellet_transport]
       │  Conservative spherical shell flux (r=0 symmetry, r=R_p interface)
       ▼
Surface Kinetics & Active Sites [ma_stoichiometry]
       ├── Mode A: PSSA (Urmès 2019 baseline)
       └── Mode B: Dynamic Surface ODEs (Berger 2008 / Urmès 2022)
```


## 1. Parameters & Configuration `[sc_manipulated_input]`

Physical properties, geometric dimensions, operational parameters, and solver configuration are consolidated into a dataclass.

- **Bed geometry**: Reactor length $L$, bed porosity $\varepsilon_b$, tube diameter $d_t$.
- **Pellet geometry**: Particle radius $R_p$, pellet porosity $\varepsilon_p$, tortuosity $\tau_p$, density $\rho_s$.
- **Active sites**: Total site capacity $C_t \ [\text{mol sites}/\text{kg}_{cat}]$.
- **Operating**: Isothermal temperature $T \ [\text{K}]$, pressure $P \ [\text{Pa}]$, superficial gas velocity $u_s \ [\text{m/s}]$.


In [ ]:
from dataclasses import dataclass, field
from typing import Literal, Dict, Tuple
import numpy as np

@dataclass
class ReactorConfig:
    # Reactor bed geometry
    L: float = 0.5                  # [m] Reactor length
    d_tube: float = 0.005           # [m] Inner tube diameter (5 mm)
    epsilon_b: float = 0.45         # [-] Bed void fraction
    u_s: float = 0.1                # [m/s] Superficial gas velocity

    # Catalyst pellet properties
    R_p: float = 75e-6              # [m] Pellet radius (75 um, Urmes sieve mid-point)
    epsilon_p: float = 0.50         # [-] Pellet porosity
    tau_p: float = 3.0              # [-] Pellet pore tortuosity
    rho_s: float = 1550.0           # [kg/m^3] Catalyst pellet density
    C_t: float = 1.0                # [mol/kg] Total active site capacity

    # Operating conditions
    T: float = 400.0                # [K] Isothermal operating temperature
    P: float = 101325.0             # [Pa] Absolute pressure (1 atm)
    R_gas: float = 8.314462         # [J/(mol K)] Universal gas constant

    # Operational mode switches
    surface_mode: Literal["PSSA", "dynamic"] = "PSSA"
    film_resistance: bool = False   # False => c_p(R_p) = C_b (Dirichlet)

    # Spatial discretization grid
    N_z: int = 10                   # Number of axial finite-volume cells
    N_r: int = 5                    # Number of radial finite-volume shell cells

print("ReactorConfig dataclass defined successfully.")


## 2. Conservative Spatial Grid Geometry `[ph_pellet_transport]`

To guarantee machine-precision mass conservation, we use **Finite Volume (FV) cell-averaged quantities**:

### Radial Spherical Shell Volumes and Interfacial Areas
For a pellet with $N_r$ shells between $r = 0$ and $r = R_p$:
- Shell boundaries: $r_{k+1/2} = k \cdot \Delta r$ for $k = 0, \dots, N_r$.
- Shell cell volumes: $V_k = \frac{4}{3}\pi \left( r_{k+1/2}^3 - r_{k-1/2}^3 \right)$.
- Interfacial flux areas: $A_{k+1/2} = 4\pi r_{k+1/2}^2$.
- Cell midpoint (for property evaluation): $\bar{r}_k = \frac{r_{k-1/2} + r_{k+1/2}}{2}$.

### Boundary Flux Conditions
- **Center ($r = 0$, $k=0$)**: $A_{1/2} \left.\frac{\partial c_p}{\partial r}\right|_{0} = 0$ (exact zero flux by symmetry).
- **Outer Shell ($r = R_p$, $k=N_r-1$)**: $-D_e A_{shell} \left.\frac{\partial c_p}{\partial r}\right|_{R_p} = \text{flux entering bulk stream}$.


In [ ]:
class SphericalPelletGrid:
    """
    Conservative radial finite-volume staggered mesh for spherical catalyst pellet.
    """
    def __init__(self, R_p: float, N_r: int):
        self.R_p = R_p
        self.N_r = N_r
        self.dr = R_p / N_r
        
        # Face boundaries (N_r + 1)
        self.r_faces = np.linspace(0.0, R_p, N_r + 1)
        
        # Cell center radii (N_r)
        self.r_centers = 0.5 * (self.r_faces[:-1] + self.r_faces[1:])
        
        # Shell volumes V_k = (4/3)*pi*(r_{k+1}^3 - r_k^3)
        self.volumes = (4.0 / 3.0) * np.pi * (self.r_faces[1:]**3 - self.r_faces[:-1]**3)
        
        # Interface face areas A_face = 4*pi*r^2
        self.face_areas = 4.0 * np.pi * self.r_faces**2
        
        # Total pellet volume
        self.V_total = (4.0 / 3.0) * np.pi * (R_p**3)
        assert np.isclose(np.sum(self.volumes), self.V_total), "Volume summation error"

grid = SphericalPelletGrid(R_p=75e-6, N_r=5)
print(f"Pellet mesh verified: {grid.N_r} shells, total volume = {grid.V_total:.4e} m^3")
